In [13]:
import sqlite3
import pandas as pd
import numpy as np
import os

conn = sqlite3.connect("../data/nfl.db")
os.makedirs("../outputs", exist_ok=True)

# Monte Carlo tables (500 simulated seasons)
season_summary = pd.read_sql("SELECT * FROM season_simulation_summary", conn)
season_records = pd.read_sql("SELECT * FROM season_records_by_sim", conn)
playoff_seeds_500 = pd.read_sql("SELECT * FROM playoff_seeds_by_sim", conn)
playoff_results_500 = pd.read_sql("SELECT * FROM playoff_results_by_sim", conn)
all_simulated_games = pd.read_sql("SELECT * FROM all_simulated_games", conn)

# Final single-season tables (one detailed season, with player stats)
final_season_games = pd.read_sql("SELECT * FROM final_season_games", conn)
final_season_players = pd.read_sql("SELECT * FROM final_season_players", conn)
final_playoff_seeds = pd.read_sql("SELECT * FROM final_playoff_seeds", conn)
final_playoff_games = pd.read_sql("SELECT * FROM final_playoff_games", conn)
final_playoff_players = pd.read_sql("SELECT * FROM final_playoff_players", conn)

teams_full = pd.read_sql("SELECT * FROM teams", conn)
team_conf_div = teams_full[["team_id", "team_conf", "team_division"]].drop_duplicates(subset="team_id")

print("Monte Carlo tables:", season_summary.shape, season_records.shape, all_simulated_games.shape)
print("Final season tables:", final_season_games.shape, final_season_players.shape, final_playoff_games.shape)

Monte Carlo tables: (32, 5) (16000, 5) (136000, 7)
Final season tables: (272, 10) (4896, 14) (13, 6)


In [ ]:
abbr_to_id = dict(zip(teams_full["team_abbr"], teams_full["team_id"]))

team_games_ref = pd.read_sql("SELECT team_id, season, team_abbr_current FROM team_games", conn)
team_abbr_lookup = team_games_ref.sort_values("season").groupby("team_id")["team_abbr_current"].last().to_dict()

# manual overrides for relocated franchises (same fix as notebook 05 aaded chargers)
abbr_overrides = {
    2510: "LAR",              # Rams: St. Louis -> Los Angeles
    abbr_to_id["LV"]: "LV",   # Raiders: Oakland -> Las Vegas
    abbr_to_id["LAC"]: "LAC", # Chargers: San Diego -> Los Angeles
}
team_abbr_lookup.update(abbr_overrides)

print("Rams:", team_abbr_lookup[2510], " | Raiders:", team_abbr_lookup[abbr_to_id["LV"]])

Rams: LAR  | Raiders: LV


In [15]:
home_rows = all_simulated_games[["sim_id", "week", "home_id", "away_id", "home_score", "away_score"]].rename(
    columns={"home_id": "team_id", "away_id": "opp_id", "home_score": "points_for", "away_score": "points_against"})
away_rows = all_simulated_games[["sim_id", "week", "home_id", "away_id", "home_score", "away_score"]].rename(
    columns={"away_id": "team_id", "home_id": "opp_id", "away_score": "points_for", "home_score": "points_against"})
all_team_games_full = pd.concat([home_rows, away_rows], ignore_index=True)

team_stat_leaders = all_team_games_full.groupby("team_id").agg(
    avg_points_for=("points_for", "mean"),
    avg_points_against=("points_against", "mean")
).reset_index()
team_stat_leaders["avg_point_diff"] = team_stat_leaders["avg_points_for"] - team_stat_leaders["avg_points_against"]
team_stat_leaders["team_abbr"] = team_stat_leaders["team_id"].map(team_abbr_lookup)

print("=== TOP 10 OFFENSES (avg points/game, 500 sims) ===")
print(team_stat_leaders.sort_values("avg_points_for", ascending=False).head(10)[["team_abbr", "avg_points_for"]].to_string(index=False))
print("\n=== TOP 10 DEFENSES (avg points allowed/game, 500 sims) ===")
print(team_stat_leaders.sort_values("avg_points_against", ascending=True).head(10)[["team_abbr", "avg_points_against"]].to_string(index=False))

=== TOP 10 OFFENSES (avg points/game, 500 sims) ===
team_abbr  avg_points_for
      BAL       25.605412
      JAX       24.090941
      LAR       24.078706
      PHI       23.870588
      SEA       23.690588
      DET       23.640706
      DAL       23.477529
      CIN       23.467765
       GB       23.253647
      WAS       23.235176

=== TOP 10 DEFENSES (avg points allowed/game, 500 sims) ===
team_abbr  avg_points_against
      SEA           20.188588
      DEN           20.312941
      BAL           20.661059
       TB           20.842235
       KC           20.874588
      PHI           21.088941
       NE           21.126941
      BUF           21.373882
       NO           21.442941
      LAC           21.664824


In [ ]:
# division standings: merge conference/division onto the Monte Carlo summary with avg record for display
standings = season_summary.merge(team_conf_div, on="team_id")

avg_records = season_records.groupby("team_id").agg(avg_wins=("wins", "mean"), avg_losses=("losses", "mean")).reset_index()
standings = standings.merge(avg_records, on="team_id")
standings["record"] = standings["avg_wins"].round(1).astype(str) + "-" + standings["avg_losses"].round(1).astype(str)

standings = standings.sort_values(["team_conf", "team_division", "playoff_pct"], ascending=[True, True, False])

for conf in ["AFC", "NFC"]:
    print(f"\n{'='*50}\n{conf}\n{'='*50}")
    conf_data = standings[standings["team_conf"] == conf]
    for div in sorted(conf_data["team_division"].unique()):
        print(f"\n--- {div} ---")
        for _, row in conf_data[conf_data["team_division"] == div].iterrows():
            print(f"  {row['team_abbr']:<4} {row['record']:<10} Playoffs: {row['playoff_pct']:>5.1f}%   SB Win: {row['sb_win_pct']:>4.1f}%")


AFC

--- AFC East ---
  BUF  9.3-7.2    Playoffs:  66.0%   SB Win:  8.0%
  NE   8.0-8.4    Playoffs:  38.0%   SB Win:  1.4%
  MIA  7.3-9.1    Playoffs:  34.4%   SB Win:  2.2%
  NYJ  7.1-9.2    Playoffs:  29.8%   SB Win:  1.0%

--- AFC North ---
  BAL  11.1-5.3   Playoffs:  88.6%   SB Win: 11.4%
  CIN  9.0-7.4    Playoffs:  59.4%   SB Win:  3.2%
  PIT  7.3-9.1    Playoffs:  31.2%   SB Win:  1.6%
  CLE  6.5-10.0   Playoffs:  19.6%   SB Win:  0.2%

--- AFC South ---
  JAX  9.4-7.1    Playoffs:  67.0%   SB Win:  6.2%
  HOU  8.4-8.0    Playoffs:  51.6%   SB Win:  2.4%
  IND  7.1-9.3    Playoffs:  29.8%   SB Win:  0.6%
  TEN  6.9-9.6    Playoffs:  26.8%   SB Win:  1.2%

--- AFC West ---
  DEN  9.7-6.8    Playoffs:  70.4%   SB Win:  6.8%
  KC   7.8-8.6    Playoffs:  40.0%   SB Win:  5.2%
  SD   7.0-9.5    Playoffs:  26.2%   SB Win:  0.8%
  LV   6.6-9.8    Playoffs:  21.2%   SB Win:  1.2%

NFC

--- NFC East ---
  PHI  9.8-6.6    Playoffs:  67.6%   SB Win:  4.4%
  WAS  8.0-8.4    Playoffs:  40

In [17]:
lines = ["# Monte Carlo Regular Season Report", "*Based on 500 simulated regular seasons*\n"]

lines.append("\n## Top 10 Offenses (avg points scored/game)\n")
lines.append("| Team | Avg Points For |")
lines.append("|------|----------------|")
for _, row in team_stat_leaders.sort_values("avg_points_for", ascending=False).head(10).iterrows():
    lines.append(f"| {row['team_abbr']} | {row['avg_points_for']:.1f} |")

lines.append("\n## Top 10 Defenses (avg points allowed/game)\n")
lines.append("| Team | Avg Points Against |")
lines.append("|------|---------------------|")
for _, row in team_stat_leaders.sort_values("avg_points_against", ascending=True).head(10).iterrows():
    lines.append(f"| {row['team_abbr']} | {row['avg_points_against']:.1f} |")

lines.append("\n## Top 10 by Point Differential\n")
lines.append("| Team | Avg Point Diff |")
lines.append("|------|-----------------|")
for _, row in team_stat_leaders.sort_values("avg_point_diff", ascending=False).head(10).iterrows():
    lines.append(f"| {row['team_abbr']} | {row['avg_point_diff']:+.1f} |")

lines.append("\n## Division Standings (500-sim averages)\n")
for conf in ["AFC", "NFC"]:
    lines.append(f"\n### {conf}\n")
    conf_data = standings[standings["team_conf"] == conf]
    for div in sorted(conf_data["team_division"].unique()):
        lines.append(f"\n#### {div}\n")
        lines.append("| Team | Record | Playoff % | Super Bowl % |")
        lines.append("|------|--------|-----------|---------------|")
        for _, row in conf_data[conf_data["team_division"] == div].iterrows():
            lines.append(f"| {row['team_abbr']} | {row['record']} | {row['playoff_pct']:.1f}% | {row['sb_win_pct']:.1f}% |")

with open("../outputs/01_monte_carlo_regular_season.md", "w") as f:
    f.write("\n".join(lines))

print("Saved 01_monte_carlo_regular_season.md")

Saved 01_monte_carlo_regular_season.md


In [18]:
playoff_appearances = playoff_seeds_500.groupby("team_id").size().reset_index(name="playoff_appearances")
sb_win_counts = playoff_results_500["sb_winner"].value_counts().reset_index()
sb_win_counts.columns = ["team_id", "sb_wins_count"]
conf_champ_counts = pd.concat([playoff_results_500["afc_champ"], playoff_results_500["nfc_champ"]]).value_counts().reset_index()
conf_champ_counts.columns = ["team_id", "conf_champ_count"]

counts_summary = season_summary.merge(playoff_appearances, on="team_id", how="left")
counts_summary = counts_summary.merge(conf_champ_counts, on="team_id", how="left")
counts_summary = counts_summary.merge(sb_win_counts, on="team_id", how="left")
counts_summary[["playoff_appearances", "conf_champ_count", "sb_wins_count"]] = counts_summary[
    ["playoff_appearances", "conf_champ_count", "sb_wins_count"]
].fillna(0).astype(int)

lines = ["# Monte Carlo Playoff & Super Bowl Odds", "*Based on 500 simulated full seasons + playoff brackets*\n"]
lines.append("\n## Championship & Super Bowl Odds (out of 500 simulations)\n")
lines.append("| Team | Playoff % (count) | Conf Champ % (count) | Super Bowl Win % (count) |")
lines.append("|------|--------------------|-----------------------|----------------------------|")
for _, row in counts_summary.sort_values("sb_win_pct", ascending=False).iterrows():
    lines.append(f"| {row['team_abbr']} | {row['playoff_pct']:.1f}% ({row['playoff_appearances']}) | {row['conf_champ_pct']:.1f}% ({row['conf_champ_count']}) | {row['sb_win_pct']:.1f}% ({row['sb_wins_count']}) |")

with open("../outputs/02_monte_carlo_playoff_odds.md", "w") as f:
    f.write("\n".join(lines))

print("Saved 02_monte_carlo_playoff_odds.md")
print(counts_summary.sort_values("sb_win_pct", ascending=False).head(10)[["team_abbr", "playoff_pct", "conf_champ_pct", "sb_win_pct"]].to_string(index=False))

Saved 02_monte_carlo_playoff_odds.md
team_abbr  playoff_pct  conf_champ_pct  sb_win_pct
      BAL         88.6            20.0        11.4
      BUF         66.0            14.4         8.0
      DEN         70.4            13.0         6.8
      SEA         70.6            13.8         6.8
      JAX         67.0            11.4         6.2
       KC         40.0             6.8         5.2
      DET         57.2            10.0         5.0
      PHI         67.6            10.4         4.4
      LAR         56.4            10.6         4.2
       TB         53.8             7.4         4.2


In [19]:
def generate_team_report(team_id, team_abbr, games_df, players_df, team_abbr_lookup):
    lines = [f"# {team_abbr} — 2026 Simulated Season\n"]

    team_games_this = games_df[(games_df["home_id"] == team_id) | (games_df["away_id"] == team_id)].sort_values("week")

    wins, losses, ties = 0, 0, 0
    lines.append("## Schedule & Results\n")
    lines.append("| Week | Matchup | Score | Result |")
    lines.append("|:----:|:-------:|:-----:|:------:|")
    for _, g in team_games_this.iterrows():
        is_home = g["home_id"] == team_id
        team_score = g["home_score"] if is_home else g["away_score"]
        opp_score = g["away_score"] if is_home else g["home_score"]
        opp_abbr = team_abbr_lookup[g["away_id"] if is_home else g["home_id"]]
        matchup = f"vs {opp_abbr}" if is_home else f"@ {opp_abbr}"

        if team_score > opp_score:
            wins += 1; result = "W"
        elif team_score < opp_score:
            losses += 1; result = "L"
        else:
            ties += 1; result = "T"

        lines.append(f"| {g['week']} | {matchup} | {team_score:.0f}-{opp_score:.0f} | {result} |")

    lines.append(f"\n**Final Record: {wins}-{losses}-{ties}**\n")

    team_players = players_df[players_df["team_id"] == team_id]
    for stat_type, header in [("passing", "Passing"), ("rushing", "Rushing"), ("receiving", "Receiving")]:
        lines.append(f"\n## {header}\n")
        lines.append("| Player | Yards | TDs |")
        lines.append("|:------:|:-----:|:---:|")
        summary = team_players[team_players["stat_type"] == stat_type].groupby("player_name").agg(
            yards=("simulated_stat", "sum"), tds=("tds", "sum")
        ).reset_index().sort_values("yards", ascending=False)
        for _, r in summary.iterrows():
            lines.append(f"| {r['player_name']} | {r['yards']:.0f} | {r['tds']:.0f} |")

    return "\n".join(lines)

print("Function defined")

Function defined


In [20]:
for conf in ["AFC", "NFC"]:
    conf_team_ids = teams_full[teams_full["team_conf"] == conf]["team_id"].drop_duplicates().tolist()
    conf_lines = [f"# 2026 NFL Simulated Season — {conf}\n"]

    for tid in conf_team_ids:
        abbr = team_abbr_lookup.get(tid, "UNK")
        conf_lines.append(generate_team_report(tid, abbr, final_season_games, final_season_players, team_abbr_lookup))
        conf_lines.append("\n---\n")

    filename = f"../outputs/03_single_season_{conf.lower()}_team_reports.md"
    with open(filename, "w") as f:
        f.write("\n".join(conf_lines))
    print(f"Saved {len(conf_team_ids)} {conf} team reports to {filename}")

Saved 16 AFC team reports to ../outputs/03_single_season_afc_team_reports.md
Saved 16 NFC team reports to ../outputs/03_single_season_nfc_team_reports.md


In [21]:
def get_box_score_text(game_id, games_df, players_df, team_abbr_lookup):
    lines = []
    g = games_df[games_df["game_id"] == game_id].iloc[0]
    home_abbr = team_abbr_lookup[g["home_id"]]
    away_abbr = team_abbr_lookup[g["away_id"]]

    lines.append(f"\n### {away_abbr} @ {home_abbr} (Week {g['week']})")
    lines.append(f"**Final: {away_abbr} {g['away_score']:.0f} - {g['home_score']:.0f} {home_abbr}**\n")

    for tid, abbr, team_score in [(g["home_id"], home_abbr, g["home_score"]), (g["away_id"], away_abbr, g["away_score"])]:
        lines.append(f"**{abbr} ({team_score:.0f} pts)**\n")
        team_stats = players_df[(players_df["game_id"] == game_id) & (players_df["team_id"] == tid)]
        team_stats = team_stats[team_stats["simulated_stat"] >= 1]

        for _, r in team_stats[team_stats["stat_type"] == "passing"].iterrows():
            lines.append(f"- {r['player_name']}: {r['completions']:.0f}/{r['attempts']:.0f}, {r['simulated_stat']:.0f} yds, {r['tds']:.0f} TD")
        for _, r in team_stats[team_stats["stat_type"] == "rushing"].sort_values("simulated_stat", ascending=False).iterrows():
            lines.append(f"- {r['player_name']}: {r['carries']:.0f} car, {r['simulated_stat']:.0f} yds, {r['tds']:.0f} TD")
        for _, r in team_stats[team_stats["stat_type"] == "receiving"].sort_values("simulated_stat", ascending=False).iterrows():
            lines.append(f"- {r['player_name']}: {r['receptions']:.0f} rec, {r['simulated_stat']:.0f} yds, {r['tds']:.0f} TD")
        lines.append("")

    return "\n".join(lines)

print("Function defined")

Function defined


In [22]:
all_game_ids = final_season_games.sort_values("week")["game_id"].tolist()
full_season_lines = ["# 2026 NFL Simulated Season — Full Regular Season Box Scores\n"]

current_week = None
for game_id in all_game_ids:
    week = final_season_games[final_season_games["game_id"] == game_id]["week"].iloc[0]
    if week != current_week:
        full_season_lines.append(f"\n---\n## Week {week}\n")
        current_week = week
    full_season_lines.append(get_box_score_text(game_id, final_season_games, final_season_players, team_abbr_lookup))

with open("../outputs/04_single_season_full_box_scores.md", "w") as f:
    f.write("\n".join(full_season_lines))

size_kb = os.path.getsize("../outputs/04_single_season_full_box_scores.md") / 1024
print(f"Saved {len(all_game_ids)} box scores to 04_single_season_full_box_scores.md ({size_kb:.1f} KB)")

Saved 272 box scores to 04_single_season_full_box_scores.md (190.1 KB)


In [23]:
def top_leaders(players_df, stat_type, team_abbr_lookup):
    leaders = players_df[players_df["stat_type"] == stat_type].groupby(["player_name", "team_id"]).agg(
        yards=("simulated_stat", "sum"), tds=("tds", "sum")
    ).reset_index().sort_values("yards", ascending=False)
    leaders["team_abbr"] = leaders["team_id"].map(team_abbr_lookup)
    return leaders

qb_leaders = top_leaders(final_season_players, "passing", team_abbr_lookup)
rb_leaders = top_leaders(final_season_players, "rushing", team_abbr_lookup)
wr_leaders = top_leaders(final_season_players, "receiving", team_abbr_lookup)

lines = ["# Single Season — Top 32 Player Leaders\n"]
for leaders_df, header in [(qb_leaders, "Passing"), (rb_leaders, "Rushing"), (wr_leaders, "Receiving")]:
    lines.append(f"\n## {header} (Top 32)\n")
    lines.append("| Rank | Player | Team | Yards | TDs |")
    lines.append("|:----:|:------:|:----:|:-----:|:---:|")
    for i, (_, row) in enumerate(leaders_df.head(32).iterrows(), 1):
        lines.append(f"| {i} | {row['player_name']} | {row['team_abbr']} | {row['yards']:.0f} | {row['tds']:.0f} |")

with open("../outputs/05_single_season_top32_leaders.md", "w") as f:
    f.write("\n".join(lines))

print("Saved 05_single_season_top32_leaders.md")
print(qb_leaders.head(5)[["player_name", "team_abbr", "yards", "tds"]].to_string(index=False))

Saved 05_single_season_top32_leaders.md
     player_name team_abbr  yards  tds
      Joe Burrow       CIN 4399.6   30
    Kyler Murray       MIN 4386.2   25
Matthew Stafford       LAR 4382.2   21
     Sam Darnold       SEA 4371.7   20
     C.J. Stroud       HOU 4357.4   19


In [24]:
def get_playoff_box_score_text(row, players_df, team_abbr_lookup):
    lines = []
    team_a_abbr = team_abbr_lookup[row["team_a"]]
    team_b_abbr = team_abbr_lookup[row["team_b"]]
    winner_abbr = team_abbr_lookup[row["winner"]]

    lines.append(f"\n### {row['round']}: {team_a_abbr} vs {team_b_abbr}")
    lines.append(f"**Final: {team_a_abbr} {row['score_a']:.0f} - {row['score_b']:.0f} {team_b_abbr}**")
    lines.append(f"**Winner: {winner_abbr}**\n")

    for tid, abbr, team_score in [(row["team_a"], team_a_abbr, row["score_a"]), (row["team_b"], team_b_abbr, row["score_b"])]:
        lines.append(f"**{abbr} ({team_score:.0f} pts)**\n")
        team_stats = players_df[(players_df["round"] == row["round"]) & (players_df["team_id"] == tid)]
        team_stats = team_stats[team_stats["simulated_stat"] >= 1]

        for _, r in team_stats[team_stats["stat_type"] == "passing"].iterrows():
            lines.append(f"- {r['player_name']}: {r['completions']:.0f}/{r['attempts']:.0f}, {r['simulated_stat']:.0f} yds, {r['tds']:.0f} TD")
        for _, r in team_stats[team_stats["stat_type"] == "rushing"].sort_values("simulated_stat", ascending=False).iterrows():
            lines.append(f"- {r['player_name']}: {r['carries']:.0f} car, {r['simulated_stat']:.0f} yds, {r['tds']:.0f} TD")
        for _, r in team_stats[team_stats["stat_type"] == "receiving"].sort_values("simulated_stat", ascending=False).iterrows():
            lines.append(f"- {r['player_name']}: {r['receptions']:.0f} rec, {r['simulated_stat']:.0f} yds, {r['tds']:.0f} TD")
        lines.append("")

    return "\n".join(lines)

playoff_lines = ["# Single Season — Full Playoff Box Scores\n"]
for _, game_row in final_playoff_games.iterrows():
    playoff_lines.append(get_playoff_box_score_text(game_row, final_playoff_players, team_abbr_lookup))
    playoff_lines.append("\n---")

with open("../outputs/06_single_season_playoff_box_scores.md", "w") as f:
    f.write("\n".join(playoff_lines))

print("Saved 06_single_season_playoff_box_scores.md")
print("\nSuper Bowl result:")
print(final_playoff_games[final_playoff_games["round"] == "Super Bowl"][["team_a", "team_b", "score_a", "score_b", "winner"]])

Saved 06_single_season_playoff_box_scores.md

Super Bowl result:
    team_a  team_b  score_a  score_b  winner
12    4400    2510       10       26    2510
